In [1]:
from datasets import load_dataset, Dataset
from trl import SFTConfig, SFTTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer
import random

/workspace/llm-autoformalization/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PROMPT = """
Please autoformalize the following problem in Lean 4.
{nl_problem}
"""

def prepare_dataset(dataset, size=None):
    if size is None:
        size = len(dataset)

    data = []
    for sample in dataset:
        nl_problem = sample["natural_language_statement"]
        formal_problem = sample["formal_statement"]

        prompt = PROMPT.format(nl_problem=nl_problem)
        completion = f"```lean\n{formal_problem}\n```"

        data.append({
            "prompt": [{"role": "user", "content": prompt}],
            "completion": [{"role": "assistant", "content": completion}]
        })

    data = data[:size]
    
    return Dataset.from_list(data)

In [3]:
dsname = "internlm/Lean-Workbook"
ds = load_dataset(dsname)

In [4]:
dataset = prepare_dataset(ds["train"], size=5000)
split_dataset = dataset.train_test_split(test_size=0.2, seed=42)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

In [5]:
model_name = "Qwen/Qwen3.5-0.8B"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="cuda"
)

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 320/320 [00:00<00:00, 1702.85it/s]


In [6]:
config = SFTConfig(
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    warmup_steps=len(train_dataset)*0.05,
    lr_scheduler_type="cosine",

    num_train_epochs=1,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
)

In [7]:
train_dataset[0]

{'prompt': [{'role': 'user',
   'content': '\nPlease autoformalize the following problem in Lean 4.\nFor positives $a$ , $b$ and $c$ we obtain\\n\\n$\\sum_{cyc}a\\sum_{cyc}\\frac{1}{1+a^2}\\geq\\sum_{cyc}\\frac{a^2}{1+a^2}\\sum_{cyc}\\frac{1}{a}\\Leftrightarrow\\sum_{cyc}a\\sum_{cyc}\\frac{1}{1+a^2}\\geq\\left(3-\\sum_{cyc}\\frac{1}{1+a^2}\\right)\\sum_{cyc}\\frac{1}{a}\\Leftrightarrow$\\n\\n$\\Leftrightarrow\\sum_{cyc}\\left(a+\\frac{1}{a}\\right)\\sum_{cyc}\\frac{1}{1+a^2}\\geq3\\sum_{cyc}\\frac{1}{a}\\Leftrightarrow\\sum_{cyc}\\left(\\frac{1+a^2}{a(1+b^2)}+\\frac{1+a^2}{a(1+c^2)}\\right)\\geq2\\sum_{cyc}\\frac{1}{a}\\Leftrightarrow$\\n\\n$\\Leftrightarrow\\sum_{cyc}\\left(\\frac{1+a^2}{a(1+b^2)}+\\frac{1+b^2}{b(1+a^2)}-\\frac{1}{a}-\\frac{1}{b}\\right)\\geq\\Leftrightarrow\\sum_{cyc}(a^2-b^2)\\left(\\frac{1}{a(1+b^2)}-\\frac{1}{b(1+a^2)}\\right)\\geq0\\Leftrightarrow$\\n\\n$\\Leftrightarrow\\sum_{cyc}\\frac{(a-b)^2(a+b)(ab-1)}{ab(1+a^2)(1+b^2)}\\geq0$ . Done!\n'}],
 'completion': [{

In [8]:
train = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=config
)

Tokenizing eval dataset: 100%|██████████| 1000/1000 [00:08<00:00, 116.80 examples/s]


In [9]:
train.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'pad_token_id': 248044}.


Step,Training Loss,Validation Loss
100,0.275879,0.272260
200,0.315175,0.309459
300,0.283056,0.285559
400,0.250271,0.255349
500,0.212170,0.247047
600,0.253314,0.226557
700,0.215667,0.213487
800,0.209392,0.208186
900,0.190526,0.207115
1000,0.201400,0.206961


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.77s/it]


TrainOutput(global_step=1000, training_loss=0.2550695390701294, metrics={'train_runtime': 1531.5742, 'train_samples_per_second': 2.612, 'train_steps_per_second': 0.653, 'total_flos': 2881171264737792.0, 'train_loss': 0.2550695390701294})